# Custom RAG Pre-Embedding (document_chunks_v2)
Embeds a local file using OpenAI `text-embedding-3-small` and inserts vectors into the `document_chunks_v2` table in PostgreSQL.

**No FILE_ID or COLLECTION_NAME from OpenWebUI needed** — `collection_name` here is just a label you define to namespace chunks for your own tool/MCP server.

## 1. Configuration

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

DB_URL = os.getenv("DB_URL")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
FILE_PATH  = os.getenv("FILE_PATH")

TALBLE_VERSION = "document_chunks_v3"
COLLECTION_NAME = "aramco_kb"

EMBEDDING_MODEL = "text-embedding-3-small"
VECTOR_DIM      = 1536

# 1024/128 sweet spot for dense corporate/financial text:
#   - 1024 tokens preserves full paragraphs/sections for semantic coherence
#   - 128 overlap (~12.5%) prevents context loss at chunk boundaries
CHUNK_SIZE      = 1024
CHUNK_OVERLAP   = 128

BATCH_SIZE      = 100     # embeddings per API call (max 2048)

In [5]:
import psycopg2
try:
    conn = psycopg2.connect(DB_URL, connect_timeout=10)
    print("Connected!", conn.get_dsn_parameters())
    conn.close()
except Exception as e:
    print(f"Failed: {e}")


Connected! {'user': 'postgres', 'channel_binding': 'prefer', 'connect_timeout': '10', 'dbname': 'railway', 'host': 'thomas.proxy.rlwy.net', 'port': '15444', 'options': '', 'sslmode': 'prefer', 'sslnegotiation': 'postgres', 'sslcompression': '0', 'sslcertmode': 'allow', 'sslsni': '1', 'ssl_min_protocol_version': 'TLSv1.2', 'gssencmode': 'prefer', 'krbsrvname': 'postgres', 'gssdelegation': '0', 'target_session_attrs': 'any', 'load_balance_hosts': 'disable'}


## 3. Load and chunk the file

In [6]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

# Read file — handles .txt and basic .pdf extraction
import os
ext = os.path.splitext(FILE_PATH)[1].lower()

if ext == ".pdf":
    try:
        import pypdf
    except ImportError:
        import subprocess; subprocess.run(["pip", "install", "pypdf"], check=True)
        import pypdf
    reader = pypdf.PdfReader(FILE_PATH)
    raw_text = "\n".join(page.extract_text() or "" for page in reader.pages)
else:
    with open(FILE_PATH, "r", encoding="utf-8", errors="ignore") as f:
        raw_text = f.read()

print(f"File loaded: {len(raw_text):,} characters")

# Chunk by tokens
tokens = enc.encode(raw_text)
print(f"Total tokens: {len(tokens):,}")

chunks = []
start = 0
while start < len(tokens):
    end = min(start + CHUNK_SIZE, len(tokens))
    chunk_tokens = tokens[start:end]
    chunk_text = enc.decode(chunk_tokens).strip()
    if chunk_text:
        chunks.append(chunk_text)
    start += CHUNK_SIZE - CHUNK_OVERLAP

print(f"Chunks created: {len(chunks)}")

File loaded: 10,473,409 characters
Total tokens: 2,812,047
Chunks created: 3139


## 4. Embed with OpenAI

In [7]:
import time
import json
import os
from openai import OpenAI, RateLimitError
from tqdm import tqdm
import tiktoken

client = OpenAI(api_key=OPENAI_API_KEY)
enc = tiktoken.get_encoding("cl100k_base")

PROGRESS_FILE = "embedding_progress.json"
TPM_LIMIT = 900_000  # stay 10% under the 1M hard limit

# Load saved progress if resuming
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE) as f:
        saved = json.load(f)
    all_embeddings = saved["embeddings"]
    start_batch = saved["next_batch"]
    print(f"Resuming from batch {start_batch} ({len(all_embeddings)} embeddings already done)")
else:
    all_embeddings = []
    start_batch = 0

# Sliding window to track tokens used in the last 60 seconds
token_window = []  # list of (timestamp, token_count)

def tokens_used_last_minute():
    cutoff = time.time() - 60
    while token_window and token_window[0][0] < cutoff:
        token_window.pop(0)
    return sum(t for _, t in token_window)

batch_indices = list(range(start_batch * BATCH_SIZE, len(chunks), BATCH_SIZE))

for i in tqdm(batch_indices, desc="Embedding", initial=start_batch, total=(len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]
    batch_tokens = sum(len(enc.encode(c)) for c in batch)

    # Proactive throttle: wait if adding this batch would exceed TPM limit
    while tokens_used_last_minute() + batch_tokens > TPM_LIMIT:
        time.sleep(0.5)

    # Retry loop with exponential backoff
    delay = 5
    for attempt in range(8):
        try:
            response = client.embeddings.create(input=batch, model=EMBEDDING_MODEL)
            break
        except RateLimitError:
            if attempt == 7:
                raise
            print(f"\nRate limited, waiting {delay}s (attempt {attempt+1}/8)...")
            time.sleep(delay)
            delay = min(delay * 2, 120)

    all_embeddings.extend([e.embedding for e in response.data])
    token_window.append((time.time(), batch_tokens))

    # Save progress after each batch
    batch_num = i // BATCH_SIZE + 1
    with open(PROGRESS_FILE, "w") as pf:
        json.dump({"embeddings": all_embeddings, "next_batch": batch_num}, pf)

# Clean up progress file on success
if os.path.exists(PROGRESS_FILE):
    os.remove(PROGRESS_FILE)

print(f"Embeddings done: {len(all_embeddings)} vectors of dim {len(all_embeddings[0])}")


Embedding:   0%|          | 0/32 [00:00<?, ?it/s]


Rate limited, waiting 5s (attempt 1/8)...

Rate limited, waiting 10s (attempt 2/8)...

Rate limited, waiting 20s (attempt 3/8)...

Rate limited, waiting 40s (attempt 4/8)...

Rate limited, waiting 80s (attempt 5/8)...

Rate limited, waiting 120s (attempt 6/8)...

Rate limited, waiting 120s (attempt 7/8)...


Embedding:   0%|          | 0/32 [06:49<?, ?it/s]


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

## 5. Insert into PostgreSQL

In [ ]:
import uuid, json, os
import psycopg2

conn = psycopg2.connect(DB_URL)
cur = conn.cursor()

# Create table if it doesn't exist (same schema as document_chunk for easy comparison)
cur.execute(f"""
    CREATE TABLE IF NOT EXISTS {TALBLE_VERSION} (
        id              TEXT PRIMARY KEY,
        collection_name TEXT NOT NULL,
        vector          VECTOR({VECTOR_DIM}),
        text            TEXT,
        vmetadata       JSONB
    )
""")
cur.execute(f"CREATE INDEX IF NOT EXISTS idx_{TABLE_VERSION}_collection ON {TABLE_VERSION} (collection_name)")

# Clear existing chunks for this collection (safe to re-run)
cur.execute("DELETE FROM document_chunks_v2 WHERE collection_name = %s", (COLLECTION_NAME,))
print(f"Cleared existing chunks for '{COLLECTION_NAME}'")

for i, (chunk, emb) in enumerate(zip(chunks, all_embeddings)):
    cur.execute("""
        INSERT INTO document_chunks_v2 (id, collection_name, vector, text, vmetadata)
        VALUES (%s, %s, %s::vector, %s, %s::jsonb)
        ON CONFLICT (id) DO NOTHING
    """, (
        str(uuid.uuid4()),
        COLLECTION_NAME,
        str(emb),
        chunk,
        json.dumps({"page": i, "source": os.path.basename(FILE_PATH)})
    ))

conn.commit()
cur.close()
conn.close()
print(f"Inserted {len(chunks)} chunks into 'document_chunks_v2' under collection '{COLLECTION_NAME}'")

Cleared existing chunks for 'aramco_kb'
Inserted 2597 chunks into 'document_chunks_v2' under collection 'aramco_kb'


## 6. Verify

In [ ]:
conn = psycopg2.connect(DB_URL)
cur = conn.cursor()
cur.execute(
    f"SELECT COUNT(*) FROM {TABLE_VERSION} WHERE collection_name = %s",
    (COLLECTION_NAME,)
)
count = cur.fetchone()[0]
cur.close()
conn.close()

print(f"Chunks in {TABLE_VERSION} for '{COLLECTION_NAME}': {count}")
print("Done — use this table in your custom tool/MCP server for RAG queries.")

Chunks in document_chunks_v2 for 'aramco_kb': 2597
Done — use this table in your custom tool/MCP server for RAG queries.


## Note: Query-time embedding
When your tool/MCP server receives a user question, embed it with the same model:
```python
client.embeddings.create(input=[user_query], model="text-embedding-3-small")
```
Then cosine-search against `document_chunks_v2`.

In [ ]:
conn = psycopg2.connect(DB_URL)
cur = conn.cursor()
cur.execute(f"SELECT COUNT(*) FROM {TABLE_VERSION} WHERE collection_name = %s", (COLLECTION_NAME,))
print(cur.fetchone()[0])

2597
